<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-absolute-level-anchor/67B_postE50_absolute_level_anchor_rsna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 67B · Post-E50 — Absolute Lumbar Level Anchor (RSNA, Stage A + Stage B design)

**Nuevo roadmap (esta revisión):** 67A demostró que disc detection + relative ordering funcionan
sobre SPIDER, pero el naming anatómico absoluto (`L1-L2...L5-S1`) sigue `UNAVAILABLE_FROM_DATASET_REFERENCE`
porque SPIDER solo provee identidad relativa (bottom-up). El absolute anchor pasa a ser requisito
previo para axial cluster-level pairing. Roadmap renombrado (conceptual, no se tocan notebooks
anteriores):

- **67A** = SPIDER relative localization validation (cerrado, commit final en
  `research/post-e50-spider-level-anchor`).
- **67B** (este notebook) = absolute lumbar level anchor.
- **67C** = axial cluster-level pairing (futuro, bloqueado hasta que 67B produzca un anchor).

## Objetivo de 67B

Resolver de manera reproducible y evaluable: `predicted relative disc instances -> absolute lumbar
levels (L1-L2, L2-L3, L3-L4, L4-L5, L5-S1)`, **sin usar ground truth para decidir la respuesta
durante inferencia** (GT se usa exclusivamente para entrenamiento/evaluación, nunca para elegir el
signo del eje PCA, decidir inferior/superior, o corregir una predicción).

SPIDER no puede ser referencia universal de absolute level (sus labels de máscara son relativos,
bottom-up). El candidato principal de dataset es **RSNA 2024 Lumbar Spine Degenerative
Classification** (aquí "RSNA LumbarDISC"), esperado bajo
`<PFI_DRIVE_ROOT>/data/RSNA_LUMBAR_DISC` o equivalente local.

## Contexto real ya disponible en este repo (citado, no asumido)

Existe trabajo previo (línea de producto **P10.6-AI**, rama `enzo/p10-6-ai-rsna-findings`, no
mergeada a esta rama) que ya auditó la estructura real de este mismo dataset RSNA para un objetivo
distinto (clasificación de hallazgos degenerativos, no localización de nivel). Se **cita** aquí como
contexto -- `docs/P10_6_RSNA_FINDINGS_SCOPE.md` y `ai_service/pfi_ai_service/training/rsna_preflight.py`
en esa rama -- pero **no se importa código de esa rama** (no está mergeada, no es una dependencia de
67B) y **no se asume que su estructura observada siga siendo válida** sin que este propio notebook la
re-verifique de forma independiente cuando el dataset esté disponible. La cita documenta qué
esperar, no qué se da por sentado:
- CSVs reales: `train.csv` (wide, un severity-string por columna `{condition}_{level}`, por
  `study_id`), `train_label_coordinates.csv` (long: `study_id, series_id, instance_number,
  condition, level, x, y` en píxeles), `train_series_descriptions.csv` (`study_id, series_id,
  series_description`), `train_images/` (DICOM).
- Niveles reales observados: `L1-L2 ... L5-S1` (formato crudo con `/` en vez de `-`, requiere
  normalización).
- Test oficial: `test_images/`, `test_series_descriptions.csv`, `sample_submission.csv` -- **nunca**
  se abren/inventarían/usan.
- Licencia: uso académico/no comercial únicamente.

## Prohibiciones de esta revisión

NO se toca `main`, Backend, Frontend, product runtime, Notebook 67, Notebook 67A (cerrado),
checkpoint `cf11...`, `AUTOMATIC_DISC_LOCALIZATION_VALIDATED`. NO se entrena nada (ni pathology
models ni el disc-level classifier). NO axial pairing. NO cross-frame registration. NO pathology
grading. NO tuning de 67A. NO se usa test (SPIDER ni RSNA). NO commit. NO push.

Rama: `research/post-e50-absolute-level-anchor` (creada desde el commit final de
`research/post-e50-spider-level-anchor`).


In [1]:
# --- Setup: environment detection ---
import hashlib
import json
import os
import re
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

EXECUTION_START = time.time()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("ENVIRONMENT:", "COLAB" if IN_COLAB else "LOCAL")


IN_COLAB: True
ENVIRONMENT: COLAB


## STEP 1 — Mount Google Drive (Colab only)


In [2]:
DRIVE_MOUNTED = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = Path('/content/drive/MyDrive').is_dir()
    print("Drive mounted:", DRIVE_MOUNTED)
else:
    print("Not in Colab: skipping Drive mount. Local execution uses environment variables instead.")


Mounted at /content/drive
Drive mounted: True


## Configuración explícita de sesión Colab

No se depende de ningún clon de repo en Drive ni de `ai_service` importado dinámicamente. Este
notebook es autocontenido -- las únicas funciones reutilizadas de 67A/Notebook 67 (checkpoint
resolver, geometría física) se redefinen aquí explícitamente citando su fuente, no se importan.


In [3]:
if IN_COLAB:
    os.environ["PFI_DRIVE_ROOT"] = "/content/drive/MyDrive/PFI_MVP"
    os.environ["PFI_RSNA_ROOT"] = "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC"
    os.environ["PFI_POST_E50_SAGITTAL_CHECKPOINT"] = "/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best_cf11dcc0.pt"
    for key in ("PFI_DRIVE_ROOT", "PFI_RSNA_ROOT", "PFI_POST_E50_SAGITTAL_CHECKPOINT"):
        print(f"{key} = {os.environ[key]}")
else:
    print("Local execution: Colab Drive environment variables not forced.")


PFI_DRIVE_ROOT = /content/drive/MyDrive/PFI_MVP
PFI_RSNA_ROOT = /content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC
PFI_POST_E50_SAGITTAL_CHECKPOINT = /content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best_cf11dcc0.pt


## STEP 2 — Workspace local del notebook

Mismo patrón que 67A: workspace efímero en `/content` dentro de Colab (solo para
artifacts/reportes temporales); checkout local real fuera de Colab.


In [4]:
def _find_repo_root_local(start: Path) -> Path | None:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "artifacts").exists():
            return candidate
    return None


_local_repo = _find_repo_root_local(Path.cwd())

if IN_COLAB:
    REPO_ROOT = Path("/content/pfi_post_e50_67B_workspace")
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    REPO_ROOT_SOURCE = "ephemeral_colab_workspace"
else:
    REPO_ROOT = _local_repo or Path.cwd()
    REPO_ROOT_SOURCE = "auto_detected_local" if _local_repo is not None else "cwd_local"

print("Workspace source:", REPO_ROOT_SOURCE)
print("Workspace root (folder name only, no absolute path persisted):", REPO_ROOT.name)


Workspace source: ephemeral_colab_workspace
Workspace root (folder name only, no absolute path persisted): pfi_post_e50_67B_workspace


## STEP 3 — Configure `PFI_DRIVE_ROOT` / `PFI_RSNA_ROOT`

Verificación de **contenido real** (no solo nombre) antes de aceptar `RSNA_ROOT`. No se asume
`train_label_coordinates.csv` -- se comprueba que exista.


In [5]:
PFI_DRIVE_ROOT_ENV = os.environ.get("PFI_DRIVE_ROOT")
PFI_DRIVE_ROOT = Path(PFI_DRIVE_ROOT_ENV) if PFI_DRIVE_ROOT_ENV else None
print("PFI_DRIVE_ROOT exists:", PFI_DRIVE_ROOT.is_dir() if PFI_DRIVE_ROOT else None)

RSNA_EXPECTED_FILES = ["train.csv", "train_label_coordinates.csv", "train_series_descriptions.csv", "train_images"]
RSNA_OFFICIAL_TEST_MARKERS = ["test_images", "test_series_descriptions.csv", "sample_submission.csv"]


def _rsna_content_report(path: Path) -> dict:
    if not path.is_dir():
        return {"rsna_root_exists": False, "files_found": {}, "official_test_present": False}
    files_found = {name: (path / name).exists() for name in RSNA_EXPECTED_FILES}
    official_test_present = any((path / marker).exists() for marker in RSNA_OFFICIAL_TEST_MARKERS)
    return {"rsna_root_exists": True, "files_found": files_found, "official_test_present": official_test_present}


RSNA_ROOT_ENV = os.environ.get("PFI_RSNA_ROOT")
RSNA_ROOT = Path(RSNA_ROOT_ENV) if RSNA_ROOT_ENV else None
rsna_content_report = _rsna_content_report(RSNA_ROOT) if RSNA_ROOT else {"rsna_root_exists": False, "files_found": {}, "official_test_present": False}

# RSNA_AVAILABLE requires the 3 train CSVs to actually exist (not just the folder or train_images/).
RSNA_AVAILABLE = bool(
    rsna_content_report["rsna_root_exists"]
    and rsna_content_report["files_found"].get("train.csv")
    and rsna_content_report["files_found"].get("train_label_coordinates.csv")
    and rsna_content_report["files_found"].get("train_series_descriptions.csv")
)
OFFICIAL_TEST_PRESENT = bool(rsna_content_report["official_test_present"])
OFFICIAL_TEST_ACCESSED = False  # structurally never set True anywhere in this notebook

print(json.dumps(rsna_content_report, indent=2))
print("RSNA_AVAILABLE:", RSNA_AVAILABLE)
print("officialTestPresent:", OFFICIAL_TEST_PRESENT, "officialTestAccessed:", OFFICIAL_TEST_ACCESSED)
if not RSNA_AVAILABLE:
    print("Expected outside Colab / without Drive mounted -- not a failure by itself. See GATE_A below.")


PFI_DRIVE_ROOT exists: True
{
  "rsna_root_exists": true,
  "files_found": {
    "train.csv": true,
    "train_label_coordinates.csv": true,
    "train_series_descriptions.csv": true,
    "train_images": true
  },
  "official_test_present": false
}
RSNA_AVAILABLE: True
officialTestPresent: False officialTestAccessed: False


## Privacy helpers, allowed write scope, git identity

Mismo diseño que 67A: `safe_write_text` restringe escritura a un allowlist explícito y rechaza
contenido con rutas locales (`C:\Users\`, `/Users/`). `study_id`/`series_id` de RSNA se tratan como
identificadores pseudónimos -- se hashean con `opaque_id` antes de persistir en cualquier artifact
o reporte versionable, igual que en 67A/66/65.


In [6]:
def opaque_id(raw_value) -> str:
    return hashlib.sha256(str(raw_value).encode("utf-8")).hexdigest()[:12]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ANCHOR_DIR = REPO_ROOT / "artifacts" / "post_e50" / "absolute_level_anchor"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"
ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
DRIVE_RESULTS_DIR = (PFI_DRIVE_ROOT / "results" / "post_e50" / "67B") if PFI_DRIVE_ROOT is not None else None
DRIVE_METRICS_DIR = (PFI_DRIVE_ROOT / "metrics" / "post_e50" / "67B") if PFI_DRIVE_ROOT is not None else None
DRIVE_FIGURES_DIR = (PFI_DRIVE_ROOT / "figures" / "post_e50" / "67B") if PFI_DRIVE_ROOT is not None else None
if PFI_DRIVE_ROOT is not None:
    ALLOWED_WRITE_ROOTS.extend([DRIVE_RESULTS_DIR, DRIVE_METRICS_DIR, DRIVE_FIGURES_DIR])

warnings: list[str] = []
limitations: list[str] = []
FORBIDDEN_IDENTIFIER_FIELDS = ("PatientName", "PatientID", "PatientBirthDate", "InstitutionName", "AccessionNumber")


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS if root is not None):
        raise RuntimeError(f"Refusing to write outside allowed trees: {path}")
    if "C:\\Users\\" in content or "/Users/" in content:
        raise RuntimeError(f"Refusing to persist a local filesystem path into {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def run_git(*args: str) -> str | None:
    try:
        result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception:
        return None


if IN_COLAB:
    GIT_BRANCH = "ephemeral-colab-workspace"
    GIT_COMMIT = None
else:
    GIT_BRANCH = run_git("branch", "--show-current")
    GIT_COMMIT = run_git("rev-parse", "HEAD")

GENERATED_AT = datetime.now(timezone.utc).isoformat()
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)


GIT_BRANCH: ephemeral-colab-workspace
GIT_COMMIT: None


## STEP 4 — Dependencias (torch/SimpleITK/scipy/pydicom), comprobación explícita


In [7]:
def _check_importable(module_name: str) -> str | None:
    try:
        module = __import__(module_name)
        return getattr(module, "__version__", "unknown_version")
    except ImportError:
        return None


torch_probe = _check_importable("torch")
sitk_probe = _check_importable("SimpleITK")
scipy_probe = _check_importable("scipy")
pydicom_probe = _check_importable("pydicom")

print("torch:", torch_probe)
print("SimpleITK:", sitk_probe if sitk_probe else "NOT INSTALLED")
print("scipy:", scipy_probe if scipy_probe else "NOT INSTALLED")
print("pydicom:", pydicom_probe if pydicom_probe else "NOT INSTALLED")
NEEDS_SITK = sitk_probe is None
NEEDS_SCIPY = scipy_probe is None
NEEDS_PYDICOM = pydicom_probe is None
if NEEDS_SITK or NEEDS_SCIPY or NEEDS_PYDICOM:
    print("Run the next cell explicitly to install missing dependencies (not automatic).")


torch: 2.11.0+cpu
SimpleITK: NOT INSTALLED
scipy: 1.16.3
pydicom: NOT INSTALLED
Run the next cell explicitly to install missing dependencies (not automatic).


In [8]:
if NEEDS_SITK:
    print("Installing SimpleITK...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "SimpleITK"])
if NEEDS_SCIPY:
    print("Installing scipy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scipy"])
if NEEDS_PYDICOM:
    print("Installing pydicom...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pydicom"])

import torch
import SimpleITK as sitk
import scipy
import pydicom

DEPENDENCY_VERSIONS = {
    "torch_version": torch.__version__,
    "simpleitk_version": sitk.Version_VersionString(),
    "scipy_version": scipy.__version__,
    "pydicom_version": pydicom.__version__,
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(DEPENDENCY_VERSIONS, indent=2, default=str))


Installing SimpleITK...
Installing pydicom...
{
  "torch_version": "2.11.0+cpu",
  "simpleitk_version": "2.5.6",
  "scipy_version": "1.16.3",
  "pydicom_version": "3.0.2",
  "cuda_available": false
}


## STAGE A — RSNA absolute-level data audit (GATE A) — inspección real, sin asumir

`GATE_A_RSNA_dataset_structure = PASS` solo si las 3 CSV reales existen Y se pueden leer sin error.
No se asume ningún nombre de columna: se listan las columnas reales encontradas y se comparan
contra las esperadas (citadas arriba), marcando explícitamente cualquier discrepancia.


In [9]:
GATE_A_RSNA_dataset_structure = "NOT_RUN"
train_df = pd.DataFrame()
coords_df = pd.DataFrame()
series_desc_df = pd.DataFrame()
csv_schema_report = {}

if RSNA_AVAILABLE:
    try:
        train_df = pd.read_csv(RSNA_ROOT / "train.csv", dtype={"study_id": "string"})
        coords_df = pd.read_csv(RSNA_ROOT / "train_label_coordinates.csv", dtype={"study_id": "string", "series_id": "string"})
        series_desc_df = pd.read_csv(RSNA_ROOT / "train_series_descriptions.csv", dtype={"study_id": "string", "series_id": "string"})
        GATE_A_RSNA_dataset_structure = "PASS"
    except Exception as exc:
        GATE_A_RSNA_dataset_structure = "FAIL"
        warnings.append(f"RSNA CSV read failed: {exc}")
else:
    warnings.append("STAGE A NOT_RUN: RSNA dataset not available in this run (expected outside Colab / without Drive mounted).")

def _csv_shape_report(df: pd.DataFrame) -> dict:
    return {
        "shape": [int(df.shape[0]), int(df.shape[1])],
        "columns": list(df.columns),
        "null_counts": {str(c): int(df[c].isna().sum()) for c in df.columns},
    }


if GATE_A_RSNA_dataset_structure == "PASS":
    csv_schema_report = {
        "train.csv": _csv_shape_report(train_df),
        "train_label_coordinates.csv": _csv_shape_report(coords_df),
        "train_series_descriptions.csv": _csv_shape_report(series_desc_df),
        "unique_study_id": {
            "train.csv": int(train_df["study_id"].nunique()) if "study_id" in train_df.columns else None,
            "train_label_coordinates.csv": int(coords_df["study_id"].nunique()) if "study_id" in coords_df.columns else None,
            "train_series_descriptions.csv": int(series_desc_df["study_id"].nunique()) if "study_id" in series_desc_df.columns else None,
        },
        "unique_series_id": {
            "train_label_coordinates.csv": int(coords_df["series_id"].nunique()) if "series_id" in coords_df.columns else None,
            "train_series_descriptions.csv": int(series_desc_df["series_id"].nunique()) if "series_id" in series_desc_df.columns else None,
        },
        "unique_instance_annotations": int(len(coords_df)) if len(coords_df) else 0,
        "csv_sha256": {
            "train.csv": sha256_file(RSNA_ROOT / "train.csv"),
            "train_label_coordinates.csv": sha256_file(RSNA_ROOT / "train_label_coordinates.csv"),
            "train_series_descriptions.csv": sha256_file(RSNA_ROOT / "train_series_descriptions.csv"),
        },
    }
    print(json.dumps(csv_schema_report, indent=2, default=str))
print()
print("GATE_A_RSNA_dataset_structure:", GATE_A_RSNA_dataset_structure)


{
  "train.csv": {
    "shape": [
      1975,
      26
    ],
    "columns": [
      "study_id",
      "spinal_canal_stenosis_l1_l2",
      "spinal_canal_stenosis_l2_l3",
      "spinal_canal_stenosis_l3_l4",
      "spinal_canal_stenosis_l4_l5",
      "spinal_canal_stenosis_l5_s1",
      "left_neural_foraminal_narrowing_l1_l2",
      "left_neural_foraminal_narrowing_l2_l3",
      "left_neural_foraminal_narrowing_l3_l4",
      "left_neural_foraminal_narrowing_l4_l5",
      "left_neural_foraminal_narrowing_l5_s1",
      "right_neural_foraminal_narrowing_l1_l2",
      "right_neural_foraminal_narrowing_l2_l3",
      "right_neural_foraminal_narrowing_l3_l4",
      "right_neural_foraminal_narrowing_l4_l5",
      "right_neural_foraminal_narrowing_l5_s1",
      "left_subarticular_stenosis_l1_l2",
      "left_subarticular_stenosis_l2_l3",
      "left_subarticular_stenosis_l3_l4",
      "left_subarticular_stenosis_l4_l5",
      "left_subarticular_stenosis_l5_s1",
      "right_subarticular_stenosi

## Level / condition / coordinate schema audit — valores reales, no asumidos (`rsna_level_reference_audit.json`)

Se derivan `condition`/`level` reales de `train_label_coordinates.csv` (columnas `condition`,
`level`, verificadas contra las columnas reales del CSV leído, no hardcodeadas ciegamente) y se
normaliza `level` (`L4/L5` -> `L4-L5`) con una función pura y self-tested -- **no se asume** que el
separador sea `/`, se detecta empíricamente. El valor raw original **se conserva siempre** junto al
normalizado (`level_raw` + `level_normalized`, columnas separadas, nunca se sobrescribe el original).

**Terminología (esta revisión):** cada fila de `train_label_coordinates.csv` es un
`ABSOLUTE_LEVEL_REFERENCE_POINT` -- un punto anatómico usado por RSNA para anclar un finding a un
nivel. **No** se llama `disc_centroid_ground_truth`: no hay evidencia todavía de que coincida con el
centro geométrico del disco (eso se investiga empíricamente en Stage C/D, nunca se asume aquí).


In [10]:
CANONICAL_LEVELS = ["L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1"]


def normalize_level_value(raw) -> str | None:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    text = str(raw).strip().upper().replace("/", "-").replace("_", "-")
    return text if text in CANONICAL_LEVELS else None


# Synthetic self-test (independent of whether RSNA is available):
assert normalize_level_value("L4/L5") == "L4-L5", "normalize_level_value self-test FAILED (slash)"
assert normalize_level_value("l4-l5") == "L4-L5", "normalize_level_value self-test FAILED (lowercase)"
assert normalize_level_value("L4_L5") == "L4-L5", "normalize_level_value self-test FAILED (underscore)"
assert normalize_level_value("not_a_level") is None, "normalize_level_value self-test FAILED (unrecognized)"
assert normalize_level_value(None) is None, "normalize_level_value self-test FAILED (None)"
print("normalize_level_value: synthetic self-test PASSED (slash/underscore/case variants, unrecognized, None).")

level_schema_audit = {"level_column_present": False, "distinct_raw_values": [], "normalized_values": [], "unrecognized_raw_values": []}
condition_schema_audit = {"condition_column_present": False, "distinct_values": []}
coordinate_schema_audit = {"x_present": False, "y_present": False, "instance_number_present": False}
series_description_schema_audit = {"column_present": False, "distinct_values": []}

if GATE_A_RSNA_dataset_structure == "PASS" and len(coords_df):
    level_schema_audit["level_column_present"] = "level" in coords_df.columns
    if level_schema_audit["level_column_present"]:
        coords_df = coords_df.copy()
        coords_df["level_raw"] = coords_df["level"]
        coords_df["level_normalized"] = coords_df["level_raw"].apply(normalize_level_value)
        raw_levels = sorted(v for v in coords_df["level_raw"].dropna().unique())
        level_schema_audit["distinct_raw_values"] = [str(v) for v in raw_levels]
        normalized = {normalize_level_value(v) for v in raw_levels}
        level_schema_audit["normalized_values"] = sorted(v for v in normalized if v is not None)
        level_schema_audit["unrecognized_raw_values"] = [str(v) for v in raw_levels if normalize_level_value(v) is None]

    condition_schema_audit["condition_column_present"] = "condition" in coords_df.columns
    if condition_schema_audit["condition_column_present"]:
        condition_schema_audit["distinct_values"] = sorted(str(v) for v in coords_df["condition"].dropna().unique())

    coordinate_schema_audit["x_present"] = "x" in coords_df.columns
    coordinate_schema_audit["y_present"] = "y" in coords_df.columns
    coordinate_schema_audit["instance_number_present"] = "instance_number" in coords_df.columns
else:
    warnings.append("Level/condition/coordinate schema audit NOT_RUN: GATE_A did not PASS in this run.")

if GATE_A_RSNA_dataset_structure == "PASS" and "series_description" in series_desc_df.columns:
    series_description_schema_audit["column_present"] = True
    series_description_schema_audit["distinct_values"] = sorted(str(v) for v in series_desc_df["series_description"].dropna().unique())

rsna_level_reference_audit = {
    "level_schema_audit": level_schema_audit, "condition_schema_audit": condition_schema_audit,
    "coordinate_schema_audit": coordinate_schema_audit, "series_description_schema_audit": series_description_schema_audit,
}
print(json.dumps(rsna_level_reference_audit, indent=2, default=str))

# absolute_level_reference_available: YES only if we ACTUALLY observed real canonical levels
# tied to study/series/instance/coordinate in this run's data -- never assumed. This is a
# provisional tabular-reference signal only; it does NOT imply coordinate physical parity is
# validated (that is GATE_D/GATE_E, evaluated separately against real DICOM below).
if GATE_A_RSNA_dataset_structure != "PASS":
    absolute_level_reference_available = "NOT_RUN"
elif (level_schema_audit["level_column_present"] and len(level_schema_audit["normalized_values"]) == len(CANONICAL_LEVELS)
      and coordinate_schema_audit["x_present"] and coordinate_schema_audit["y_present"] and coordinate_schema_audit["instance_number_present"]):
    absolute_level_reference_available = "YES"
elif level_schema_audit["level_column_present"] and len(level_schema_audit["normalized_values"]) > 0:
    absolute_level_reference_available = "PARTIAL"
else:
    absolute_level_reference_available = "NO"

print()
print("absolute_level_reference_available (provisional, tabular-reference only):", absolute_level_reference_available)
print("This does NOT mean coordinate physical parity is validated -- see GATE_D/GATE_E below.")
if absolute_level_reference_available == "NO":
    print("STOP: no usable absolute anatomical level reference found in this run's data. Reporting as blocker, not fabricating the experiment.")


normalize_level_value: synthetic self-test PASSED (slash/underscore/case variants, unrecognized, None).
{
  "level_schema_audit": {
    "level_column_present": true,
    "distinct_raw_values": [
      "L1/L2",
      "L2/L3",
      "L3/L4",
      "L4/L5",
      "L5/S1"
    ],
    "normalized_values": [
      "L1-L2",
      "L2-L3",
      "L3-L4",
      "L4-L5",
      "L5-S1"
    ],
    "unrecognized_raw_values": []
  },
  "condition_schema_audit": {
    "condition_column_present": true,
    "distinct_values": [
      "Left Neural Foraminal Narrowing",
      "Left Subarticular Stenosis",
      "Right Neural Foraminal Narrowing",
      "Right Subarticular Stenosis",
      "Spinal Canal Stenosis"
    ]
  },
  "coordinate_schema_audit": {
    "x_present": true,
    "y_present": true,
    "instance_number_present": true
  },
  "series_description_schema_audit": {
    "column_present": true,
    "distinct_values": [
      "Axial T2",
      "Sagittal T1",
      "Sagittal T2/STIR"
    ]
  }
}



## Dataset inventory — studies / series / sequence candidates (patient-level split unit)

Split futuro a nivel `study_id` (única unidad -- no se separan slices/series de un mismo study
entre train/validation). Se deriva `sequence_role` de `series_description` de forma explícita y
auditable (no se clasifica una serie por contener una letra aislada).


In [11]:
def classify_series_description(raw: str) -> str:
    text = str(raw).strip().lower()
    if "t2" in text and ("stir" in text or "sag" in text):
        if "sag" in text:
            return "sagittal_t2_stir"
    if "sag" in text and "t1" in text:
        return "sagittal_t1"
    if "sag" in text and "t2" in text:
        return "sagittal_t2_stir"
    if "ax" in text and "t2" in text:
        return "axial_t2"
    return "unclassified"


# Synthetic self-test (independent of RSNA availability):
assert classify_series_description("Sagittal T2/STIR") == "sagittal_t2_stir", "classify_series_description self-test FAILED (sag t2/stir)"
assert classify_series_description("Sagittal T1") == "sagittal_t1", "classify_series_description self-test FAILED (sag t1)"
assert classify_series_description("Axial T2") == "axial_t2", "classify_series_description self-test FAILED (axial t2)"
assert classify_series_description("Localizer") == "unclassified", "classify_series_description self-test FAILED (unclassified)"
print("classify_series_description: synthetic self-test PASSED (sagittal T2/STIR, sagittal T1, axial T2, unclassified).")

dataset_inventory_summary = {
    "studies_total": 0, "series_total": 0, "sequence_role_counts": {},
    "studies_with_all_five_canonical_levels": 0, "studies_missing_levels": 0,
    "duplicate_label_rows": 0, "malformed_coordinate_rows": 0,
    "missing_series_id_rows": 0, "missing_instance_number_rows": 0,
    "studies_with_five_levels_by_condition": {}, "studies_with_five_levels_sagittal_t2_stir": 0,
}
rsna_series_inventory_rows = []

if GATE_A_RSNA_dataset_structure == "PASS":
    dataset_inventory_summary["studies_total"] = int(series_desc_df["study_id"].nunique()) if "study_id" in series_desc_df else 0
    dataset_inventory_summary["series_total"] = int(series_desc_df["series_id"].nunique()) if "series_id" in series_desc_df else 0
    if "series_description" in series_desc_df.columns:
        roles = series_desc_df["series_description"].apply(classify_series_description)
        dataset_inventory_summary["sequence_role_counts"] = roles.value_counts().to_dict()
        for _, srow in series_desc_df.assign(sequence_role=roles).iterrows():
            rsna_series_inventory_rows.append({
                "study_id_opaque": opaque_id(srow.get("study_id")), "series_id_opaque": opaque_id(srow.get("series_id")),
                "series_description_raw": srow.get("series_description"), "sequence_role": srow.get("sequence_role"),
            })

    if level_schema_audit["level_column_present"] and "study_id" in coords_df.columns:
        dataset_inventory_summary["missing_series_id_rows"] = int(coords_df["series_id"].isna().sum()) if "series_id" in coords_df.columns else len(coords_df)
        dataset_inventory_summary["missing_instance_number_rows"] = int(coords_df["instance_number"].isna().sum()) if "instance_number" in coords_df.columns else len(coords_df)

        per_study_levels = coords_df.groupby("study_id")["level_normalized"].apply(lambda s: set(v for v in s if v is not None))
        dataset_inventory_summary["studies_with_all_five_canonical_levels"] = int(sum(1 for levels in per_study_levels if levels == set(CANONICAL_LEVELS)))
        dataset_inventory_summary["studies_missing_levels"] = int(sum(1 for levels in per_study_levels if levels != set(CANONICAL_LEVELS)))
        dataset_inventory_summary["duplicate_label_rows"] = int(coords_df.duplicated(subset=[c for c in ["study_id", "series_id", "condition", "level_normalized"] if c in coords_df.columns]).sum())
        if coordinate_schema_audit["x_present"] and coordinate_schema_audit["y_present"]:
            malformed = coords_df[(coords_df["x"].isna()) | (coords_df["y"].isna()) | (coords_df["x"] < 0) | (coords_df["y"] < 0)]
            dataset_inventory_summary["malformed_coordinate_rows"] = int(len(malformed))

        if condition_schema_audit["condition_column_present"]:
            for cond in condition_schema_audit["distinct_values"]:
                cond_levels = coords_df[coords_df["condition"] == cond].groupby("study_id")["level_normalized"].apply(lambda s: set(v for v in s if v is not None))
                dataset_inventory_summary["studies_with_five_levels_by_condition"][str(cond)] = int(sum(1 for lv in cond_levels if lv == set(CANONICAL_LEVELS)))

        # Sagittal T2/STIR-specific count: requires joining coords_df.series_id against series_desc_df's
        # sagittal_t2_stir series (a level annotation only "counts" toward this if its series_id is
        # actually classified as sagittal_t2_stir in train_series_descriptions.csv).
        if "series_description" in series_desc_df.columns and "series_id" in coords_df.columns:
            sag_t2_series_ids = set(series_desc_df.loc[roles == "sagittal_t2_stir", "series_id"]) if "series_description" in series_desc_df.columns else set()
            sag_t2_coords = coords_df[coords_df["series_id"].isin(sag_t2_series_ids)]
            sag_t2_levels_per_study = sag_t2_coords.groupby("study_id")["level_normalized"].apply(lambda s: set(v for v in s if v is not None))
            dataset_inventory_summary["studies_with_five_levels_sagittal_t2_stir"] = int(sum(1 for lv in sag_t2_levels_per_study if lv == set(CANONICAL_LEVELS)))
else:
    warnings.append("Dataset inventory NOT_RUN: GATE_A did not PASS in this run.")

print(json.dumps(dataset_inventory_summary, indent=2, default=str))


classify_series_description: synthetic self-test PASSED (sagittal T2/STIR, sagittal T1, axial T2, unclassified).
{
  "studies_total": 1975,
  "series_total": 6294,
  "sequence_role_counts": {
    "axial_t2": 2340,
    "sagittal_t1": 1980,
    "sagittal_t2_stir": 1974
  },
  "studies_with_all_five_canonical_levels": 1974,
  "studies_missing_levels": 0,
  "duplicate_label_rows": 0,
  "malformed_coordinate_rows": 0,
  "missing_series_id_rows": 0,
  "missing_instance_number_rows": 0,
  "studies_with_five_levels_by_condition": {
    "Left Neural Foraminal Narrowing": 1972,
    "Left Subarticular Stenosis": 1799,
    "Right Neural Foraminal Narrowing": 1971,
    "Right Subarticular Stenosis": 1804,
    "Spinal Canal Stenosis": 1899
  },
  "studies_with_five_levels_sagittal_t2_stir": 1898
}


## Sagittal T2/STIR + Spinal Canal Stenosis — primary reference table (candidato para baseline)

**Hipótesis a auditar, no asumida universalmente:** para el primer baseline 67B, `condition ==
"Spinal Canal Stenosis"` combinado con `series_description == "Sagittal T2/STIR"` puede dar un
punto por nivel en una serie sagital central, geométricamente alineada con el segmentador sagital
67A. Se audita **primero** por study: cantidad de series Sagittal T2/STIR, cantidad de coordenadas
Spinal Canal Stenosis, niveles presentes, duplicados/faltantes -- antes de asumir que esta
combinación es la referencia principal.


In [12]:
PRIMARY_REFERENCE_CONDITION = "Spinal Canal Stenosis"
PRIMARY_REFERENCE_SEQUENCE_ROLE = "sagittal_t2_stir"
sag_t2_canal_stenosis_reference_rows = []
sag_t2_canal_stenosis_audit = {"studies_with_sag_t2_series": 0, "studies_with_canal_stenosis_coords": 0, "studies_with_both": 0}

if GATE_A_RSNA_dataset_structure == "PASS" and level_schema_audit["level_column_present"] and "series_description" in series_desc_df.columns:
    roles_all = series_desc_df["series_description"].apply(classify_series_description)
    sag_t2_rows = series_desc_df.assign(sequence_role=roles_all)
    sag_t2_rows = sag_t2_rows[sag_t2_rows["sequence_role"] == PRIMARY_REFERENCE_SEQUENCE_ROLE]
    sag_t2_series_by_study = sag_t2_rows.groupby("study_id")["series_id"].apply(list)

    canal_coords = coords_df[coords_df.get("condition", pd.Series(dtype=object)) == PRIMARY_REFERENCE_CONDITION] if "condition" in coords_df.columns else pd.DataFrame()

    sag_t2_canal_stenosis_audit["studies_with_sag_t2_series"] = int(sag_t2_rows["study_id"].nunique())
    sag_t2_canal_stenosis_audit["studies_with_canal_stenosis_coords"] = int(canal_coords["study_id"].nunique()) if len(canal_coords) else 0
    common_studies = set(sag_t2_rows["study_id"]) & set(canal_coords["study_id"]) if len(canal_coords) else set()
    sag_t2_canal_stenosis_audit["studies_with_both"] = len(common_studies)

    for study_id in sorted(common_studies):
        sag_series_ids = set(sag_t2_series_by_study.get(study_id, []))
        study_canal_coords = canal_coords[(canal_coords["study_id"] == study_id) & (canal_coords["series_id"].isin(sag_series_ids))]
        levels_seen = set()
        for _, r in study_canal_coords.iterrows():
            level_norm = r.get("level_normalized")
            status = "ok"
            if level_norm is None:
                status = "unrecognized_level"
            elif level_norm in levels_seen:
                status = "duplicate_level"
            levels_seen.add(level_norm)
            sag_t2_canal_stenosis_reference_rows.append({
                "study_id_opaque": opaque_id(study_id), "sag_t2_series_id_opaque": opaque_id(r.get("series_id")),
                "level_raw": r.get("level_raw"), "level_normalized": level_norm,
                "instance_number": r.get("instance_number"), "x": r.get("x"), "y": r.get("y"),
                "reference_condition": PRIMARY_REFERENCE_CONDITION, "status": status,
            })
        missing_levels = set(CANONICAL_LEVELS) - {lv for lv in levels_seen if lv is not None}
        for missing in sorted(missing_levels):
            sag_t2_canal_stenosis_reference_rows.append({
                "study_id_opaque": opaque_id(study_id), "sag_t2_series_id_opaque": None,
                "level_raw": None, "level_normalized": missing, "instance_number": None, "x": None, "y": None,
                "reference_condition": PRIMARY_REFERENCE_CONDITION, "status": "missing_level",
            })
else:
    warnings.append("Sagittal T2/STIR + Spinal Canal Stenosis reference table NOT_RUN: GATE_A did not PASS or required columns missing in this run.")

print(json.dumps(sag_t2_canal_stenosis_audit, indent=2, default=str))
print(f"sag_t2_canal_stenosis_reference_rows: {len(sag_t2_canal_stenosis_reference_rows)} rows (this hypothesis is audited, NOT assumed universal)")


{
  "studies_with_sag_t2_series": 1974,
  "studies_with_canal_stenosis_coords": 1974,
  "studies_with_both": 1974
}
sag_t2_canal_stenosis_reference_rows: 9870 rows (this hypothesis is audited, NOT assumed universal)


## GATE C — split leakage audit (study_id-level, reproducible internal split)

Reutiliza el mismo principio de 65/66/67A: nunca separar contenido de un mismo `study_id` entre
train/validation/holdout interno. `RSNA_INTERNAL_TEST_LOCKED = True` estructuralmente bloquea el
uso del holdout interno (a definir en una iteración futura) y del test oficial de Kaggle.


In [13]:
def compute_study_level_leakage_audit(train_ids: set, validation_ids: set, holdout_ids: set) -> dict:
    train_val_overlap = train_ids & validation_ids
    train_holdout_overlap = train_ids & holdout_ids
    val_holdout_overlap = validation_ids & holdout_ids
    return {
        "train_validation_overlap_count": len(train_val_overlap),
        "train_holdout_overlap_count": len(train_holdout_overlap),
        "validation_holdout_overlap_count": len(val_holdout_overlap),
        "leakage_free": len(train_val_overlap) == 0 and len(train_holdout_overlap) == 0 and len(val_holdout_overlap) == 0,
    }


# Synthetic self-test (independent of RSNA availability):
_clean = compute_study_level_leakage_audit({"a", "b"}, {"c", "d"}, {"e"})
_dirty = compute_study_level_leakage_audit({"a", "b"}, {"b", "c"}, {"e"})
assert _clean["leakage_free"] is True, "compute_study_level_leakage_audit self-test FAILED (expected clean)"
assert _dirty["leakage_free"] is False and _dirty["train_validation_overlap_count"] == 1, "compute_study_level_leakage_audit self-test FAILED (expected dirty)"
print("compute_study_level_leakage_audit: synthetic self-test PASSED (clean split + overlapping split).")

def build_reproducible_study_split(study_ids: list, seed: int = 2026, train_frac: float = 0.7, validation_frac: float = 0.15) -> dict:
    # Deterministic: sorted input + seeded shuffle -> same split every run, no slice/series ever
    # crosses a study_id boundary (the split unit is study_id itself).
    ordered = sorted(study_ids)
    rng = np.random.default_rng(seed)
    shuffled = list(ordered)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * train_frac)
    n_val = int(n * validation_frac)
    return {
        "train": sorted(shuffled[:n_train]),
        "validation": sorted(shuffled[n_train:n_train + n_val]),
        "internal_test": sorted(shuffled[n_train + n_val:]),
    }


# Synthetic self-test (independent of RSNA availability): deterministic + leakage-free + covers all ids
_synthetic_ids = [f"study_{i}" for i in range(20)]
_split_a = build_reproducible_study_split(_synthetic_ids)
_split_b = build_reproducible_study_split(_synthetic_ids)
assert _split_a == _split_b, "build_reproducible_study_split self-test FAILED (not deterministic)"
_all_assigned = set(_split_a["train"]) | set(_split_a["validation"]) | set(_split_a["internal_test"])
assert _all_assigned == set(_synthetic_ids), "build_reproducible_study_split self-test FAILED (not all ids covered)"
_audit = compute_study_level_leakage_audit(set(_split_a["train"]), set(_split_a["validation"]), set(_split_a["internal_test"]))
assert _audit["leakage_free"] is True, "build_reproducible_study_split self-test FAILED (leakage detected in own split)"
print("build_reproducible_study_split: synthetic self-test PASSED (deterministic, full coverage, leakage-free).")

RSNA_INTERNAL_TEST_LOCKED = True
GATE_C_split_leakage = "NOT_RUN"
rsna_split_manifest = {"train": [], "validation": [], "internal_test": []}
rsna_split_counts = {"train": 0, "validation": 0, "internal_test": 0}

if GATE_A_RSNA_dataset_structure == "PASS" and dataset_inventory_summary["studies_total"] > 0:
    all_study_ids = list(set(series_desc_df["study_id"].dropna().unique())) if "study_id" in series_desc_df.columns else []
    real_split = build_reproducible_study_split(all_study_ids)
    audit_result = compute_study_level_leakage_audit(set(real_split["train"]), set(real_split["validation"]), set(real_split["internal_test"]))
    GATE_C_split_leakage = "PASS" if audit_result["leakage_free"] else "FAIL"
    # Manifest persists ONLY opaque hashed ids -- never raw study_id.
    rsna_split_manifest = {split_name: [opaque_id(sid) for sid in ids] for split_name, ids in real_split.items()}
    rsna_split_counts = {split_name: len(ids) for split_name, ids in real_split.items()}
    print(json.dumps({"leakage_audit": audit_result, "split_counts": rsna_split_counts}, indent=2))
else:
    warnings.append("GATE_C split leakage audit NOT_RUN: GATE_A did not PASS or no studies found in this run.")

print("RSNA_INTERNAL_TEST_LOCKED:", RSNA_INTERNAL_TEST_LOCKED, "(internal_test split is created for manifest completeness but NEVER opened/used in this notebook)")
print("GATE_C_split_leakage:", GATE_C_split_leakage)


compute_study_level_leakage_audit: synthetic self-test PASSED (clean split + overlapping split).
build_reproducible_study_split: synthetic self-test PASSED (deterministic, full coverage, leakage-free).
{
  "leakage_audit": {
    "train_validation_overlap_count": 0,
    "train_holdout_overlap_count": 0,
    "validation_holdout_overlap_count": 0,
    "leakage_free": true
  },
  "split_counts": {
    "train": 1382,
    "validation": 296,
    "internal_test": 297
  }
}
RSNA_INTERNAL_TEST_LOCKED: True (internal_test split is created for manifest completeness but NEVER opened/used in this notebook)
GATE_C_split_leakage: PASS


## STAGE B — Series / coordinate geometry audit (GATE E) — annotation pixel -> physical XYZ

Para cada annotation de nivel, resolver el DICOM exacto (`study_id`+`series_id`+`instance_number`)
y transformar `(x, y)` píxel -> índice DICOM continuo -> XYZ físico paciente, usando
`PixelSpacing`/`ImagePositionPatient`/`ImageOrientationPatient` reales (fórmula DICOM estándar,
la misma clase de transformación que Notebook 66/67, no una nueva convención). GT centroid (nivel
RSNA) se transforma con la misma fórmula para poder compararse en el mismo espacio físico.
**No se usa cross-frame registration en esta etapa** -- se trabaja dentro de la misma sagittal
series, o en geometría que pueda demostrarse compatible.


In [14]:
def dicom_pixel_to_patient_xyz(column_index: float, row_index: float, pixel_spacing: tuple[float, float],
                                image_position_patient: tuple[float, float, float],
                                image_orientation_patient: tuple[float, float, float, float, float, float]) -> np.ndarray:
    # Standard DICOM formula (PS3.3 C.7.6.2.1.1; same class of transform as Notebook 66/67's
    # pixel_to_patient_xyz). RSNA train_label_coordinates.csv convention: x = column_index,
    # y = row_index (explicit, not assumed silently -- this is why the params are named
    # column_index/row_index here rather than generic x_px/y_px).
    # PixelSpacing = [row_spacing, column_spacing] (DICOM tag order: spacing BETWEEN rows first).
    # ImageOrientationPatient = [row_cosines(3), column_cosines(3)]: row_cosines = direction of
    # increasing COLUMN index (moving along a row); column_cosines = direction of increasing ROW
    # index (moving along a column).
    # P = IPP + column_index * column_spacing * row_cosines + row_index * row_spacing * column_cosines
    row_cosines = np.array(image_orientation_patient[0:3], dtype=np.float64)
    column_cosines = np.array(image_orientation_patient[3:6], dtype=np.float64)
    origin = np.array(image_position_patient, dtype=np.float64)
    row_spacing, column_spacing = pixel_spacing
    return origin + column_index * column_spacing * row_cosines + row_index * row_spacing * column_cosines


def resolve_dicom_instance_path(train_images_root: Path, study_id: str, series_id: str, instance_number: int) -> Path | None:
    series_dir = train_images_root / str(study_id) / str(series_id)
    if not series_dir.is_dir():
        return None
    candidate = series_dir / f"{instance_number}.dcm"
    if candidate.is_file():
        return candidate
    # Fallback: some RSNA exports do not zero-pad / do not match InstanceNumber to filename directly.
    matches = list(series_dir.glob(f"*{instance_number}*.dcm"))
    return matches[0] if matches else None


# --- Synthetic self-test: axis-aligned identity case (independent of real DICOM files) ---
_identity_xyz = dicom_pixel_to_patient_xyz(
    column_index=10.0, row_index=20.0, pixel_spacing=(1.0, 1.0),
    image_position_patient=(0.0, 0.0, 0.0),
    image_orientation_patient=(1.0, 0.0, 0.0, 0.0, 1.0, 0.0),
)
assert np.allclose(_identity_xyz, [10.0, 20.0, 0.0]), "dicom_pixel_to_patient_xyz self-test FAILED (identity case)"
print("dicom_pixel_to_patient_xyz: synthetic self-test PASSED (identity orientation/spacing/origin case).")

# --- Synthetic self-test: non-trivial spacing + offset origin, verified against manual arithmetic ---
_manual_xyz = np.array([100.0, -50.0, 25.0]) + 5.0 * 0.5 * np.array([0.0, 1.0, 0.0]) + 7.0 * 0.7 * np.array([1.0, 0.0, 0.0])
_computed_xyz = dicom_pixel_to_patient_xyz(
    column_index=5.0, row_index=7.0, pixel_spacing=(0.7, 0.5),
    image_position_patient=(100.0, -50.0, 25.0),
    image_orientation_patient=(0.0, 1.0, 0.0, 1.0, 0.0, 0.0),
)
assert np.allclose(_computed_xyz, _manual_xyz), "dicom_pixel_to_patient_xyz self-test FAILED (offset+spacing case)"
print("dicom_pixel_to_patient_xyz: synthetic self-test PASSED (offset origin + non-unit spacing, verified against manual arithmetic).")

# --- Cross-validation against SimpleITK's own geometry engine (independent implementation) ---
_sitk_check_image = sitk.Image(64, 64, 1, sitk.sitkUInt8)
_sitk_spacing = (0.5, 0.7, 1.0)  # SimpleITK spacing order is (x=column_spacing, y=row_spacing, z) -- NOT the DICOM PixelSpacing=[row,col] order
_sitk_check_image.SetSpacing(_sitk_spacing)
_sitk_check_image.SetOrigin((100.0, -50.0, 25.0))
_sitk_check_image.SetDirection((0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0))
_sitk_xyz = np.array(_sitk_check_image.TransformContinuousIndexToPhysicalPoint((5.0, 7.0, 0.0)))
assert np.allclose(_sitk_xyz, _manual_xyz, atol=1e-6), "dicom_pixel_to_patient_xyz self-test FAILED (disagrees with SimpleITK's independent geometry engine)"
print("dicom_pixel_to_patient_xyz: cross-validated against SimpleITK TransformContinuousIndexToPhysicalPoint (independent implementation) -- PASSED.")


def check_xy_inside_frame(column_index: float, row_index: float, rows: int, columns: int) -> bool:
    return (0 <= column_index < columns) and (0 <= row_index < rows)


def check_geometry_finite(*values: float) -> bool:
    return all(np.isfinite(v) for v in values)


def check_orientation_valid(image_orientation_patient: tuple) -> bool:
    if len(image_orientation_patient) != 6:
        return False
    row_cosines = np.array(image_orientation_patient[0:3], dtype=np.float64)
    column_cosines = np.array(image_orientation_patient[3:6], dtype=np.float64)
    row_norm_ok = abs(np.linalg.norm(row_cosines) - 1.0) < 1e-2
    col_norm_ok = abs(np.linalg.norm(column_cosines) - 1.0) < 1e-2
    orthogonal_ok = abs(float(np.dot(row_cosines, column_cosines))) < 1e-2
    return bool(row_norm_ok and col_norm_ok and orthogonal_ok)


assert check_xy_inside_frame(10, 10, 64, 64) is True, "check_xy_inside_frame self-test FAILED (inside)"
assert check_xy_inside_frame(-1, 10, 64, 64) is False, "check_xy_inside_frame self-test FAILED (negative column)"
assert check_xy_inside_frame(10, 100, 64, 64) is False, "check_xy_inside_frame self-test FAILED (row out of bounds)"
assert check_geometry_finite(1.0, 2.0, 3.0) is True, "check_geometry_finite self-test FAILED (finite)"
assert check_geometry_finite(1.0, float("nan")) is False, "check_geometry_finite self-test FAILED (nan)"
assert check_orientation_valid((1.0, 0.0, 0.0, 0.0, 1.0, 0.0)) is True, "check_orientation_valid self-test FAILED (identity)"
assert check_orientation_valid((1.0, 0.0, 0.0, 1.0, 0.0, 0.0)) is False, "check_orientation_valid self-test FAILED (parallel, not orthogonal)"
print("check_xy_inside_frame / check_geometry_finite / check_orientation_valid: synthetic self-tests PASSED.")


dicom_pixel_to_patient_xyz: synthetic self-test PASSED (identity orientation/spacing/origin case).
dicom_pixel_to_patient_xyz: synthetic self-test PASSED (offset origin + non-unit spacing, verified against manual arithmetic).
dicom_pixel_to_patient_xyz: cross-validated against SimpleITK TransformContinuousIndexToPhysicalPoint (independent implementation) -- PASSED.
check_xy_inside_frame / check_geometry_finite / check_orientation_valid: synthetic self-tests PASSED.


## Roundtrip self-test con un DICOM sintético real (pydicom `Dataset`)

Construye un `pydicom.Dataset` sintético mínimo (sin datos de pacientes reales) con los tags DICOM
necesarios, escribe/lee de disco, y confirma que la función de transformación arriba usa
exactamente los mismos campos que un archivo DICOM real expondría vía `pydicom`.

**Corrección de semántica de gates (esta revisión):** este self-test y el cross-check contra
SimpleITK de la celda anterior confirman que el **método** (`dicom_pixel_to_patient_xyz`) es
correcto -- **no** confirman paridad contra RSNA real. Por eso sus resultados se guardan como
sub-audits separados (`coordinate_mapping_formula_selftest`,
`simpleitk_physical_crosscheck`), **nunca** como `GATE_D`/`GATE_E`. Antes de correr contra RSNA
real en Colab, `GATE_D_series_annotation_mapping` y `GATE_E_coordinate_physical_parity` deben
permanecer `NOT_RUN` -- solo pueden pasar a `PASS` cuando se resuelven `study_id`+`series_id`+
`instance_number` reales a un DICOM real, `x`/`y` caen dentro del frame real, la metadata DICOM
real es válida, el XYZ físico se calcula sobre esos datos reales, y no hay inconsistencias
relevantes en la cohorte auditada (ver celda "Mapear cada annotation a su DICOM real" más abajo).


In [15]:
def _build_synthetic_dicom_dataset(pixel_spacing, image_position_patient, image_orientation_patient, rows=64, columns=64) -> "pydicom.Dataset":
    from pydicom.dataset import Dataset, FileMetaDataset
    from pydicom.uid import ExplicitVRLittleEndian, generate_uid

    file_meta = FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = generate_uid()
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    ds = Dataset()
    ds.file_meta = file_meta
    ds.is_little_endian = True
    ds.is_implicit_VR = False
    ds.Rows = rows
    ds.Columns = columns
    ds.PixelSpacing = [str(pixel_spacing[0]), str(pixel_spacing[1])]
    ds.ImagePositionPatient = [str(v) for v in image_position_patient]
    ds.ImageOrientationPatient = [str(v) for v in image_orientation_patient]
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.PixelData = np.zeros((rows, columns), dtype=np.uint16).tobytes()
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 0
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    return ds


coordinate_mapping_formula_selftest = "FAIL"
try:
    import tempfile as _tempfile
    _synthetic_ds = _build_synthetic_dicom_dataset(
        pixel_spacing=(0.8, 0.6),
        image_position_patient=(12.0, -34.0, 56.0),
        image_orientation_patient=(1.0, 0.0, 0.0, 0.0, 0.9396926, 0.3420201),  # ~20 deg tilt, unit row/col vectors
    )
    with _tempfile.NamedTemporaryFile(suffix=".dcm", delete=False) as _tmp_dcm:
        _synthetic_ds.save_as(_tmp_dcm.name, enforce_file_format=True)
        _tmp_dcm_path = Path(_tmp_dcm.name)
    _loaded = pydicom.dcmread(str(_tmp_dcm_path))
    _tmp_dcm_path.unlink()

    _px_spacing = tuple(float(v) for v in _loaded.PixelSpacing)
    _ipp = tuple(float(v) for v in _loaded.ImagePositionPatient)
    _iop = tuple(float(v) for v in _loaded.ImageOrientationPatient)
    _xyz_from_file = dicom_pixel_to_patient_xyz(15.0, 22.0, _px_spacing, _ipp, _iop)
    _xyz_from_source = dicom_pixel_to_patient_xyz(15.0, 22.0, (0.8, 0.6), (12.0, -34.0, 56.0), (1.0, 0.0, 0.0, 0.0, 0.9396926, 0.3420201))
    assert np.allclose(_xyz_from_file, _xyz_from_source, atol=1e-4), "roundtrip self-test FAILED (pydicom-read tags do not reproduce source transform)"
    coordinate_mapping_formula_selftest = "PASS"
    print("Synthetic pydicom Dataset roundtrip self-test PASSED: tags read back via pydicom reproduce the exact same physical XYZ.")
except Exception as exc:
    warnings.append(f"coordinate_mapping_formula_selftest FAILED: {exc}")
    print("Synthetic pydicom roundtrip self-test FAILED:", exc)

# simpleitk_physical_crosscheck reflects the SimpleITK cross-validation self-test two cells above
# (it either raised an AssertionError there, stopping execution, or passed -- if we reach this
# point, it passed).
simpleitk_physical_crosscheck = "PASS"

print()
print("coordinate_mapping_formula_selftest:", coordinate_mapping_formula_selftest, "(method-level only -- synthetic data)")
print("simpleitk_physical_crosscheck:", simpleitk_physical_crosscheck, "(method-level only -- synthetic data, independent geometry engine)")
print("Neither sub-audit is evidence of parity against REAL RSNA DICOM -- GATE_D/GATE_E below require real data and remain NOT_RUN until then.")


Synthetic pydicom Dataset roundtrip self-test PASSED: tags read back via pydicom reproduce the exact same physical XYZ.

coordinate_mapping_formula_selftest: PASS (method-level only -- synthetic data)
simpleitk_physical_crosscheck: PASS (method-level only -- synthetic data, independent geometry engine)
Neither sub-audit is evidence of parity against REAL RSNA DICOM -- GATE_D/GATE_E below require real data and remain NOT_RUN until then.


## STEP — Mapear cada annotation a su DICOM real (solo si RSNA disponible)

Si `RSNA_AVAILABLE`: para cada fila de `train_label_coordinates.csv`, resolver el archivo DICOM real
por `study_id/series_id/instance_number.dcm` (`resolve_dicom_instance_path`, robusto a filenames no
zero-padded), leer sus tags reales (`Rows`, `Columns`, `PixelSpacing`, `ImagePositionPatient`,
`ImageOrientationPatient`, `InstanceNumber`), verificar 5 checks explícitos por fila
(`RSNA_INSTANCE_RESOLUTION`, `RSNA_XY_INSIDE_FRAME`, `RSNA_GEOMETRY_FINITE`,
`RSNA_ORIENTATION_VALID`, `RSNA_PIXEL_TO_PATIENT_XYZ`), y solo entonces transformar `(x,y)` a XYZ
físico. Si algún check falla, se registra como `unresolved`/`geometry_invalid` -- **nunca se inventa
geometría**. `GATE_D`/`GATE_E` solo pueden ser `PASS` con datos reales (nunca en base al self-test
sintético de arriba, que solo demuestra el método).


In [17]:
# ============================================================
# GATE D/E — Optimized real RSNA geometry audit
# Scope: Sagittal T2/STIR + Spinal Canal Stenosis only
# Reads each unique DICOM instance ONCE.
# ============================================================

import time

GATE_D_series_annotation_mapping = "NOT_RUN"
GATE_E_coordinate_physical_parity_real = "NOT_RUN"

annotation_geometry_rows = []

dicom_resolution_stats = {
    "all_coordinate_rows": 0,
    "primary_reference_rows": 0,
    "primary_reference_unique_studies": 0,
    "primary_reference_unique_series": 0,
    "primary_reference_unique_instances": 0,
    "unique_dicom_reads": 0,
    "direct_path_hit": 0,
    "fallback_path_hit": 0,
    "unresolved_path": 0,
    "instance_resolved": 0,
    "xy_inside_frame": 0,
    "geometry_finite": 0,
    "orientation_valid": 0,
    "xyz_computed": 0,
    "cache_hit_count": 0,
}

geometry_validation_scope = {
    "condition": "Spinal Canal Stenosis",
    "series_description": "Sagittal T2/STIR",
    "levels": ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"],
}

if (
    RSNA_AVAILABLE
    and GATE_A_RSNA_dataset_structure == "PASS"
    and len(coords_df)
    and coordinate_schema_audit["x_present"]
    and coordinate_schema_audit["y_present"]
):
    t0 = time.time()

    train_images_root = Path(
        os.environ.get(
            "PFI_RSNA_TRAIN_IMAGES",
            str(RSNA_ROOT / "train_images"),
        )
    )

    # --------------------------------------------------------
    # 1. Resolve series-description dataframe robustly
    # --------------------------------------------------------
    if "series_df" in globals():
        _series_df = series_df.copy()
    elif "series_desc_df" in globals():
        _series_df = series_desc_df.copy()
    else:
        _series_df = pd.read_csv(
            RSNA_ROOT / "train_series_descriptions.csv"
        )

    # --------------------------------------------------------
    # 2. Preserve raw level if needed
    # --------------------------------------------------------
    _coords = coords_df.copy()

    if "level_raw" not in _coords.columns:
        _coords["level_raw"] = _coords["level"]

    if "level_normalized" not in _coords.columns:
        _coords["level_normalized"] = (
            _coords["level_raw"]
            .astype(str)
            .str.replace("/", "-", regex=False)
        )

    dicom_resolution_stats["all_coordinate_rows"] = len(_coords)

    # --------------------------------------------------------
    # 3. Join coordinates to actual series descriptions
    # --------------------------------------------------------
    primary_reference_df = _coords.merge(
        _series_df[
            ["study_id", "series_id", "series_description"]
        ].drop_duplicates(),
        on=["study_id", "series_id"],
        how="left",
        validate="many_to_one",
    )

    # --------------------------------------------------------
    # 4. Restrict to the ONLY scope needed for 67B right now
    # --------------------------------------------------------
    primary_reference_df = primary_reference_df[
        (
            primary_reference_df["condition"]
            .astype(str)
            .str.strip()
            == "Spinal Canal Stenosis"
        )
        &
        (
            primary_reference_df["series_description"]
            .astype(str)
            .str.strip()
            == "Sagittal T2/STIR"
        )
        &
        (
            primary_reference_df["level_raw"]
            .astype(str)
            .isin(geometry_validation_scope["levels"])
        )
    ].copy()

    primary_reference_df = primary_reference_df[
        primary_reference_df["instance_number"].notna()
        & primary_reference_df["x"].notna()
        & primary_reference_df["y"].notna()
    ].copy()

    dicom_resolution_stats["primary_reference_rows"] = len(
        primary_reference_df
    )
    dicom_resolution_stats["primary_reference_unique_studies"] = int(
        primary_reference_df["study_id"].nunique()
    )
    dicom_resolution_stats["primary_reference_unique_series"] = int(
        primary_reference_df["series_id"].nunique()
    )

    print("Geometry validation scope:")
    print(json.dumps(geometry_validation_scope, indent=2))
    print()
    print(
        "Primary reference rows:",
        len(primary_reference_df),
    )
    print(
        "Unique studies:",
        primary_reference_df["study_id"].nunique(),
    )
    print(
        "Unique series:",
        primary_reference_df["series_id"].nunique(),
    )

    # --------------------------------------------------------
    # 5. Unique DICOM instances only
    # --------------------------------------------------------
    unique_instances_df = (
        primary_reference_df[
            ["study_id", "series_id", "instance_number"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    n_unique = len(unique_instances_df)

    dicom_resolution_stats[
        "primary_reference_unique_instances"
    ] = n_unique

    print("Unique DICOM instances to read:", n_unique)
    print()

    # --------------------------------------------------------
    # 6. Metadata cache: ONE dcmread per unique instance
    # --------------------------------------------------------
    dicom_metadata_cache = {}

    specific_tags = [
        "Rows",
        "Columns",
        "PixelSpacing",
        "ImagePositionPatient",
        "ImageOrientationPatient",
        "InstanceNumber",
    ]

    for i, inst in unique_instances_df.iterrows():
        study_id = inst["study_id"]
        series_id = inst["series_id"]
        instance_number = int(inst["instance_number"])

        key = (
            str(study_id),
            str(series_id),
            instance_number,
        )

        # Direct path first: avoids expensive glob/rglob on Drive.
        direct_path = (
            train_images_root
            / str(study_id)
            / str(series_id)
            / f"{instance_number}.dcm"
        )

        dcm_path = None
        path_resolution = None

        if direct_path.exists():
            dcm_path = direct_path
            path_resolution = "direct"
            dicom_resolution_stats["direct_path_hit"] += 1
        else:
            # Fallback only when necessary.
            dcm_path = resolve_dicom_instance_path(
                train_images_root,
                study_id,
                series_id,
                instance_number,
            )

            if dcm_path is not None:
                path_resolution = "fallback"
                dicom_resolution_stats["fallback_path_hit"] += 1
            else:
                dicom_resolution_stats["unresolved_path"] += 1

        metadata = {
            "resolved": False,
            "path_resolution": path_resolution,
            "status": "unresolved",
        }

        if dcm_path is not None:
            try:
                ds = pydicom.dcmread(
                    str(dcm_path),
                    stop_before_pixels=True,
                    specific_tags=specific_tags,
                )

                dicom_resolution_stats["unique_dicom_reads"] += 1

                required_tags = [
                    "Rows",
                    "Columns",
                    "PixelSpacing",
                    "ImagePositionPatient",
                    "ImageOrientationPatient",
                    "InstanceNumber",
                ]

                if all(hasattr(ds, tag) for tag in required_tags):
                    px_spacing = tuple(
                        float(v) for v in ds.PixelSpacing
                    )
                    ipp = tuple(
                        float(v) for v in ds.ImagePositionPatient
                    )
                    iop = tuple(
                        float(v) for v in ds.ImageOrientationPatient
                    )

                    metadata = {
                        "resolved": True,
                        "status": "ok",
                        "path_resolution": path_resolution,
                        "rows": int(ds.Rows),
                        "columns": int(ds.Columns),
                        "pixel_spacing": px_spacing,
                        "ipp": ipp,
                        "iop": iop,
                        "instance_number": int(ds.InstanceNumber),
                    }

                    dicom_resolution_stats["instance_resolved"] += 1

                else:
                    metadata["status"] = "missing_required_tags"

            except Exception as exc:
                metadata["status"] = (
                    f"read_error:{type(exc).__name__}"
                )

        dicom_metadata_cache[key] = metadata

        # Progress every 100 unique DICOMs.
        done = i + 1
        if (
            done == 1
            or done % 100 == 0
            or done == n_unique
        ):
            elapsed = time.time() - t0
            print(
                f"Geometry audit: {done}/{n_unique} "
                f"({100.0 * done / max(n_unique, 1):.1f}%) "
                f"| elapsed={elapsed:.1f}s"
            )

    # --------------------------------------------------------
    # 7. Apply cached metadata to annotation rows
    # --------------------------------------------------------
    for _, row in primary_reference_df.iterrows():
        study_id = row["study_id"]
        series_id = row["series_id"]
        instance_number = int(row["instance_number"])

        key = (
            str(study_id),
            str(series_id),
            instance_number,
        )

        meta = dicom_metadata_cache.get(key)

        row_record = {
            "study_id_opaque": opaque_id(study_id),
            "series_id_opaque": opaque_id(series_id),
            "level_raw": row.get("level_raw"),
            "level_normalized": row.get("level_normalized"),
            "condition": row.get("condition"),
            "series_description": row.get("series_description"),
            "RSNA_INSTANCE_RESOLUTION": False,
            "RSNA_XY_INSIDE_FRAME": False,
            "RSNA_GEOMETRY_FINITE": False,
            "RSNA_ORIENTATION_VALID": False,
            "RSNA_PIXEL_TO_PATIENT_XYZ": False,
            "status": "unresolved",
        }

        if meta is not None:
            dicom_resolution_stats["cache_hit_count"] += 1

        if meta and meta.get("resolved"):
            row_record["RSNA_INSTANCE_RESOLUTION"] = True

            x_col = float(row["x"])
            y_row = float(row["y"])

            px_spacing = meta["pixel_spacing"]
            ipp = meta["ipp"]
            iop = meta["iop"]

            row_record["RSNA_XY_INSIDE_FRAME"] = (
                check_xy_inside_frame(
                    x_col,
                    y_row,
                    meta["rows"],
                    meta["columns"],
                )
            )

            dicom_resolution_stats["xy_inside_frame"] += int(
                row_record["RSNA_XY_INSIDE_FRAME"]
            )

            row_record["RSNA_GEOMETRY_FINITE"] = (
                check_geometry_finite(
                    *px_spacing,
                    *ipp,
                    *iop,
                )
            )

            dicom_resolution_stats["geometry_finite"] += int(
                row_record["RSNA_GEOMETRY_FINITE"]
            )

            row_record["RSNA_ORIENTATION_VALID"] = (
                check_orientation_valid(iop)
            )

            dicom_resolution_stats["orientation_valid"] += int(
                row_record["RSNA_ORIENTATION_VALID"]
            )

            if (
                row_record["RSNA_GEOMETRY_FINITE"]
                and row_record["RSNA_ORIENTATION_VALID"]
            ):
                xyz = dicom_pixel_to_patient_xyz(
                    x_col,
                    y_row,
                    px_spacing,
                    ipp,
                    iop,
                )

                row_record[
                    "RSNA_PIXEL_TO_PATIENT_XYZ"
                ] = bool(np.all(np.isfinite(xyz)))

                if row_record["RSNA_PIXEL_TO_PATIENT_XYZ"]:
                    row_record["physical_xyz"] = xyz.tolist()
                    dicom_resolution_stats["xyz_computed"] += 1

            # D = mapping + frame validity
            mapping_ok = (
                row_record["RSNA_INSTANCE_RESOLUTION"]
                and row_record["RSNA_XY_INSIDE_FRAME"]
            )

            # E = actual geometry parity requirements
            geometry_ok = (
                mapping_ok
                and row_record["RSNA_GEOMETRY_FINITE"]
                and row_record["RSNA_ORIENTATION_VALID"]
                and row_record["RSNA_PIXEL_TO_PATIENT_XYZ"]
            )

            if geometry_ok:
                row_record["status"] = "ok"
            elif mapping_ok:
                row_record["status"] = "geometry_invalid"
            else:
                row_record["status"] = "mapping_invalid"

        elif meta:
            row_record["status"] = meta.get(
                "status",
                "unresolved",
            )

        annotation_geometry_rows.append(row_record)

    # --------------------------------------------------------
    # 8. Gates — only PRIMARY reference scope
    # --------------------------------------------------------
    total_primary = len(annotation_geometry_rows)

    mapping_ok_count = sum(
        1
        for r in annotation_geometry_rows
        if (
            r["RSNA_INSTANCE_RESOLUTION"]
            and r["RSNA_XY_INSIDE_FRAME"]
        )
    )

    geometry_ok_count = sum(
        1
        for r in annotation_geometry_rows
        if r["status"] == "ok"
    )

    if total_primary > 0:
        GATE_D_series_annotation_mapping = (
            "PASS"
            if mapping_ok_count == total_primary
            else "PARTIAL"
            if mapping_ok_count > 0
            else "FAIL"
        )

        GATE_E_coordinate_physical_parity_real = (
            "PASS"
            if geometry_ok_count == total_primary
            else "PARTIAL"
            if geometry_ok_count > 0
            else "FAIL"
        )
    else:
        GATE_D_series_annotation_mapping = "FAIL"
        GATE_E_coordinate_physical_parity_real = "FAIL"

    elapsed = time.time() - t0

    dicom_resolution_stats[
        "geometry_audit_elapsed_seconds"
    ] = elapsed

    dicom_resolution_stats[
        "annotation_rows_evaluated"
    ] = total_primary

    dicom_resolution_stats[
        "average_annotations_per_dicom"
    ] = (
        total_primary / max(n_unique, 1)
    )

    print()
    print("=== Geometry audit summary ===")
    print(json.dumps(dicom_resolution_stats, indent=2))

    print(
        f"\nMapping valid: "
        f"{mapping_ok_count}/{total_primary}"
    )

    print(
        f"Geometry valid: "
        f"{geometry_ok_count}/{total_primary}"
    )

else:
    warnings.append(
        "Primary Sagittal T2/STIR Spinal Canal Stenosis "
        "annotation-to-DICOM geometry mapping NOT_RUN."
    )

print()
print(
    "GATE_D_series_annotation_mapping:",
    GATE_D_series_annotation_mapping,
)

print(
    "GATE_E_coordinate_physical_parity (real data):",
    GATE_E_coordinate_physical_parity_real,
)

Geometry validation scope:
{
  "condition": "Spinal Canal Stenosis",
  "series_description": "Sagittal T2/STIR",
  "levels": [
    "L1/L2",
    "L2/L3",
    "L3/L4",
    "L4/L5",
    "L5/S1"
  ]
}

Primary reference rows: 9748
Unique studies: 1973
Unique series: 1973
Unique DICOM instances to read: 2521

Geometry audit: 1/2521 (0.0%) | elapsed=0.2s
Geometry audit: 100/2521 (4.0%) | elapsed=0.6s
Geometry audit: 200/2521 (7.9%) | elapsed=1.1s
Geometry audit: 300/2521 (11.9%) | elapsed=1.6s
Geometry audit: 400/2521 (15.9%) | elapsed=2.0s
Geometry audit: 500/2521 (19.8%) | elapsed=5.7s
Geometry audit: 600/2521 (23.8%) | elapsed=73.4s
Geometry audit: 700/2521 (27.8%) | elapsed=150.6s
Geometry audit: 800/2521 (31.7%) | elapsed=230.1s
Geometry audit: 900/2521 (35.7%) | elapsed=309.9s
Geometry audit: 1000/2521 (39.7%) | elapsed=381.4s
Geometry audit: 1100/2521 (43.6%) | elapsed=451.5s
Geometry audit: 1200/2521 (47.6%) | elapsed=524.9s
Geometry audit: 1300/2521 (51.6%) | elapsed=599.1s
Geometry

## STAGE C — Frozen 67A baseline on RSNA — preflight only (execution NOT run in this notebook)

Solo si Stage A y B pasan sobre datos reales: usar **exactamente** el checkpoint congelado
`cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944` y el preprocessing validado en
67A (sin reentrenar el segmenter) sobre sagittal T2/STIR de RSNA, para producir relative disc
instances + centroides físicos, y compararlos (`Hungarian`/matching físico) contra los
`ABSOLUTE_LEVEL_REFERENCE_POINT` -- **solo para evaluación**, nunca para elegir el signo del eje
PCA, decidir inferior/superior, o corregir la predicción.

Este notebook **prepara** un smoke cohort determinístico de 3 studies (siempre desde el split
`validation`, nunca `internal_test`) con Sagittal T2/STIR + los 5 absolute level reference points
presentes, pero **no ejecuta** el pipeline sobre ellos todavía -- eso queda para revisar Stage A/B
primero, como se pidió explícitamente.


In [18]:
EXPECTED_CHECKPOINT_SHA256 = "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"
GATE_F_frozen_67A_execution_on_RSNA = "NOT_RUN"
GATE_G_pred_to_absolute_GT_matching = "NOT_RUN"


def select_rsna_smoke_cohort(eligible_study_ids: list, seed: int = 2026, size: int = 3) -> list:
    ordered = sorted(eligible_study_ids)
    rng = np.random.default_rng(seed)
    shuffled = list(ordered)
    rng.shuffle(shuffled)
    return sorted(shuffled[:size])


# Synthetic self-test (independent of RSNA availability):
_cohort_a = select_rsna_smoke_cohort([f"s{i}" for i in range(50)])
_cohort_b = select_rsna_smoke_cohort([f"s{i}" for i in range(50)])
assert _cohort_a == _cohort_b and len(_cohort_a) == 3, "select_rsna_smoke_cohort self-test FAILED (expected deterministic 3-study cohort)"
print("select_rsna_smoke_cohort: synthetic self-test PASSED (deterministic 3-study selection).")

rsna_smoke_cohort_opaque = []
if GATE_A_RSNA_dataset_structure == "PASS" and GATE_C_split_leakage == "PASS" and level_schema_audit["level_column_present"]:
    validation_ids_this_run = set(real_split["validation"]) if "real_split" in dir() else set()
    # Eligible: in validation split, has Sagittal T2/STIR series, and all 5 canonical levels present
    # via the Spinal Canal Stenosis reference (same hypothesis audited above -- not re-derived here).
    eligible = set()
    if len(sag_t2_canal_stenosis_reference_rows):
        per_study_status = {}
        for r in sag_t2_canal_stenosis_reference_rows:
            per_study_status.setdefault(r["study_id_opaque"], []).append(r["status"])
        opaque_to_raw = {opaque_id(sid): sid for sid in validation_ids_this_run}
        for opaque_sid, statuses in per_study_status.items():
            if opaque_sid in opaque_to_raw and all(s == "ok" for s in statuses) and len(statuses) == len(CANONICAL_LEVELS):
                eligible.add(opaque_to_raw[opaque_sid])
    if eligible:
        smoke_cohort = select_rsna_smoke_cohort(list(eligible))
        rsna_smoke_cohort_opaque = [opaque_id(sid) for sid in smoke_cohort]
    print(f"Smoke cohort eligible pool size: {len(eligible)}; selected: {len(rsna_smoke_cohort_opaque)}")
else:
    warnings.append("RSNA smoke cohort selection NOT_RUN: prerequisites (GATE_A/GATE_C/level schema) not met in this run.")

warnings.append("STAGE C (frozen 67A baseline EXECUTION on RSNA) intentionally NOT run in this notebook -- preflight/cohort-selection only, per explicit instruction. Review Stage A/B results first.")
print("GATE_F_frozen_67A_execution_on_RSNA:", GATE_F_frozen_67A_execution_on_RSNA)
print("GATE_G_pred_to_absolute_GT_matching:", GATE_G_pred_to_absolute_GT_matching)
print("Expected checkpoint SHA-256 (reference only, not loaded in this run):", EXPECTED_CHECKPOINT_SHA256)
print("rsna_smoke_cohort (opaque study_id hashes):", rsna_smoke_cohort_opaque)


select_rsna_smoke_cohort: synthetic self-test PASSED (deterministic 3-study selection).
Smoke cohort eligible pool size: 290; selected: 3
GATE_F_frozen_67A_execution_on_RSNA: NOT_RUN
GATE_G_pred_to_absolute_GT_matching: NOT_RUN
Expected checkpoint SHA-256 (reference only, not loaded in this run): cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944
rsna_smoke_cohort (opaque study_id hashes): ['4f06df2fd53b', 'd41a396f20c6', 'ef2ff5b618cf']


## STAGE D — Absolute anchor baseline design (documentado, NO implementado ni entrenado)

Diseño planificado para una iteración futura, **no ejecutado aquí**:

- **Input:** crop sagittal alrededor de cada disc instance detectado (frozen 67A pipeline).
- **Target:** clasificación 5 clases (`L1-L2`...`L5-S1`).
- Puede incluir información geométrica normalizada del study, **nunca** `study_id`/metadata que
  produzca leakage.
- Modelo inicial: pequeño/reproducible (ej. CNN pequeña o ResNet18 si está disponible y su
  licencia es adecuada) -- **no se busca SOTA**.
- **No se entrena en esta entrega** -- ni siquiera si el dataset audit pasara, hasta revisión
  explícita del diseño de mapeo geométrico.

`ABSOLUTE_LEVEL_TRAINING_DATASET_READY` se calcula abajo, dependiente de los gates reales de esta
corrida (no se asume `YES` sin evidencia).


In [19]:
# Training-dataset readiness:
# Stage A/B can make us READY FOR STAGE C,
# but the actual absolute-level training dataset is not READY
# until frozen-67A execution + matching (GATE F/G) are validated.

stage_ab_ready_for_stage_c = (
    GATE_A_RSNA_dataset_structure == "PASS"
    and absolute_level_reference_available == "YES"
    and GATE_C_split_leakage == "PASS"
    and GATE_D_series_annotation_mapping == "PASS"
    and GATE_E_coordinate_physical_parity_real == "PASS"
)

ABSOLUTE_LEVEL_TRAINING_DATASET_READY = "NO"

if stage_ab_ready_for_stage_c:
    ABSOLUTE_LEVEL_TRAINING_DATASET_READY = "PARTIAL"

if (
    stage_ab_ready_for_stage_c
    and GATE_F_frozen_67A_execution_on_RSNA == "PASS"
    and GATE_G_pred_to_absolute_GT_matching == "PASS"
):
    ABSOLUTE_LEVEL_TRAINING_DATASET_READY = "YES"

print(
    "stage_ab_ready_for_stage_c:",
    stage_ab_ready_for_stage_c,
)

print(
    "ABSOLUTE_LEVEL_TRAINING_DATASET_READY:",
    ABSOLUTE_LEVEL_TRAINING_DATASET_READY,
)

stage_ab_ready_for_stage_c: True
ABSOLUTE_LEVEL_TRAINING_DATASET_READY: PARTIAL


## STAGE E — Future sequence consistency (documentación únicamente, sin código de decisión)

Para una iteración futura: el classifier dará logits por disc instance detectado. La secuencia se
puede resolver globalmente con una restricción `L1-L2 -> L2-L3 -> L3-L4 -> L4-L5 -> L5-S1` (sin
duplicar niveles, conservando monotonicidad), vía programación dinámica, constrained assignment, o
sequence decoding -- **sin usar GT en ese decoder**. Si la confianza es baja: `ABSTAIN`. No se
fuerza a asignar los 5 niveles si el FOV está incompleto (lección directa de 67A: SPIDER mostró FOV
con 3 a 9 discs visibles -- un decoder de secuencia absoluta debe tolerar FOV parcial, no asumir
exactamente 5). Esta sección es solo documentación -- no se implementa código de decisión aquí.


## Quality gates A-I (Stage A + Stage B de esta corrida)

Muchos gates quedan `NOT_RUN` en esta iteración -- es el resultado correcto y esperado (Stage A
localmente sin RSNA disponible, Stage C/D intencionalmente no ejecutados).


In [20]:
gate_status_67b: dict[str, str] = {
    "GATE_A_RSNA_dataset_structure": GATE_A_RSNA_dataset_structure,
    "GATE_B_absolute_level_reference": absolute_level_reference_available,
    "GATE_C_split_leakage": GATE_C_split_leakage,
    "GATE_D_series_annotation_mapping": GATE_D_series_annotation_mapping,
    # GATE_E can ONLY come from real-data validation (GATE_E_coordinate_physical_parity_real,
    # NOT_RUN unless RSNA_AVAILABLE and real DICOM checks actually ran). It must never fall back to
    # a synthetic self-test result -- those are separate sub_audits, see below.
    "GATE_E_coordinate_physical_parity": GATE_E_coordinate_physical_parity_real,
    "GATE_F_frozen_67A_execution_on_RSNA": GATE_F_frozen_67A_execution_on_RSNA,
    "GATE_G_pred_to_absolute_GT_matching": GATE_G_pred_to_absolute_GT_matching,
    "GATE_H_training_dataset_readiness": ABSOLUTE_LEVEL_TRAINING_DATASET_READY,
    "GATE_I_privacy": None,
}
sub_audits_67b = {
    "coordinate_mapping_formula_selftest": coordinate_mapping_formula_selftest,
    "simpleitk_physical_crosscheck": simpleitk_physical_crosscheck,
}
gates_67b_df = pd.DataFrame(sorted(gate_status_67b.items()), columns=["gate", "status"])
print(gates_67b_df)
print()
print(json.dumps(sub_audits_67b, indent=2))


                                  gate   status
0        GATE_A_RSNA_dataset_structure     PASS
1      GATE_B_absolute_level_reference      YES
2                 GATE_C_split_leakage     PASS
3     GATE_D_series_annotation_mapping     PASS
4    GATE_E_coordinate_physical_parity     PASS
5  GATE_F_frozen_67A_execution_on_RSNA  NOT_RUN
6  GATE_G_pred_to_absolute_GT_matching  NOT_RUN
7    GATE_H_training_dataset_readiness  PARTIAL
8                       GATE_I_privacy     None

{
  "coordinate_mapping_formula_selftest": "PASS",
  "simpleitk_physical_crosscheck": "PASS"
}


## Privacy audit (GATE I)


In [21]:
candidate_outputs_67b = {
    "csv_schema_report": json.dumps(csv_schema_report, default=str),
    "dataset_inventory_summary": json.dumps(dataset_inventory_summary, default=str),
    "annotation_geometry_rows_sample": json.dumps(annotation_geometry_rows[:5], default=str),
}
privacy_findings_67b = []
for name, text in candidate_outputs_67b.items():
    if "C:\\Users\\" in text or "/Users/" in text:
        privacy_findings_67b.append(f"{name} contains a local filesystem path")
    for forbidden_field in FORBIDDEN_IDENTIFIER_FIELDS:
        if forbidden_field in text:
            privacy_findings_67b.append(f"{name} contains forbidden identifier field {forbidden_field}")

GATE_I_privacy_67b = "PASS" if not privacy_findings_67b else "FAIL"
gate_status_67b["GATE_I_privacy"] = GATE_I_privacy_67b
gates_67b_df.loc[gates_67b_df["gate"] == "GATE_I_privacy", "status"] = GATE_I_privacy_67b
print("Privacy findings:", privacy_findings_67b if privacy_findings_67b else "(none)")
print("officialTestPresent:", OFFICIAL_TEST_PRESENT, "officialTestAccessed:", OFFICIAL_TEST_ACCESSED)
print(gates_67b_df)


Privacy findings: (none)
officialTestPresent: False officialTestAccessed: False
                                  gate   status
0        GATE_A_RSNA_dataset_structure     PASS
1      GATE_B_absolute_level_reference      YES
2                 GATE_C_split_leakage     PASS
3     GATE_D_series_annotation_mapping     PASS
4    GATE_E_coordinate_physical_parity     PASS
5  GATE_F_frozen_67A_execution_on_RSNA  NOT_RUN
6  GATE_G_pred_to_absolute_GT_matching  NOT_RUN
7    GATE_H_training_dataset_readiness  PARTIAL
8                       GATE_I_privacy     PASS


## Decisión y blockers


In [22]:
blocking_gates_67b = [g for g, s in gate_status_67b.items() if s not in ("PASS", "YES")]

if GATE_A_RSNA_dataset_structure != "PASS":
    decision_67b = "BLOCKED"
    blocker_reason = "RSNA_DATASET_NOT_AVAILABLE_IN_THIS_RUN"
elif absolute_level_reference_available == "NO":
    decision_67b = "BLOCKED"
    blocker_reason = "NO_USABLE_ABSOLUTE_LEVEL_REFERENCE_FOUND"
elif ABSOLUTE_LEVEL_TRAINING_DATASET_READY == "YES":
    decision_67b = "STAGE_A_B_PASSED_READY_FOR_STAGE_C"
    blocker_reason = None
else:
    decision_67b = "PARTIAL"
    blocker_reason = "SEE_BLOCKING_GATES"

print("Decision:", decision_67b)
print("Blocker reason:", blocker_reason)
print("Blocking gates:", blocking_gates_67b)
print("ABSOLUTE_LEVEL_TRAINING_DATASET_READY:", ABSOLUTE_LEVEL_TRAINING_DATASET_READY)


Decision: PARTIAL
Blocker reason: SEE_BLOCKING_GATES
Blocking gates: ['GATE_F_frozen_67A_execution_on_RSNA', 'GATE_G_pred_to_absolute_GT_matching', 'GATE_H_training_dataset_readiness']
ABSOLUTE_LEVEL_TRAINING_DATASET_READY: PARTIAL


## Artefactos locales (schema-only, sin datos de RSNA)


In [23]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


environment_record_67b = {
    "generated_at": GENERATED_AT, "in_colab": IN_COLAB,
    "dependency_versions": DEPENDENCY_VERSIONS if "DEPENDENCY_VERSIONS" in dir() else None,
    "rsna_available": RSNA_AVAILABLE, "official_test_present": OFFICIAL_TEST_PRESENT,
    "official_test_accessed": OFFICIAL_TEST_ACCESSED,
    "expected_checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "sub_audits": sub_audits_67b,
    "dicom_resolution_stats": dicom_resolution_stats if "dicom_resolution_stats" in dir() else None,
}
safe_write_text(ANCHOR_DIR / "post_e50_67B_environment.json", json.dumps(environment_record_67b, indent=2, default=_json_default, ensure_ascii=False))
safe_write_text(ANCHOR_DIR / "post_e50_67B_quality_gates.json", json.dumps({"gates": gate_status_67b, "sub_audits": sub_audits_67b, "warnings": warnings}, indent=2, default=str, ensure_ascii=False))

# Local schema-only artifacts requested explicitly (never contain raw study_id/series_id).
safe_write_text(ANCHOR_DIR / "rsna_dataset_inventory.json", json.dumps(dataset_inventory_summary, indent=2, default=_json_default, ensure_ascii=False))
safe_write_text(ANCHOR_DIR / "rsna_level_reference_audit.json", json.dumps(rsna_level_reference_audit if "rsna_level_reference_audit" in dir() else {}, indent=2, default=_json_default, ensure_ascii=False))
if len(rsna_series_inventory_rows):
    pd.DataFrame(rsna_series_inventory_rows).to_csv(ANCHOR_DIR / "rsna_series_inventory.csv", index=False)
if len(sag_t2_canal_stenosis_reference_rows):
    pd.DataFrame(sag_t2_canal_stenosis_reference_rows).to_csv(ANCHOR_DIR / "sag_t2_canal_stenosis_reference_table.csv", index=False)
safe_write_text(ANCHOR_DIR / "post_e50_67B_split_manifest.json", json.dumps({"split_counts": rsna_split_counts, "manifest_opaque": rsna_split_manifest, "rsna_internal_test_locked": RSNA_INTERNAL_TEST_LOCKED}, indent=2, default=_json_default, ensure_ascii=False))
safe_write_text(ANCHOR_DIR / "post_e50_67B_smoke_cohort.json", json.dumps({"smoke_cohort_opaque": rsna_smoke_cohort_opaque}, indent=2, ensure_ascii=False))

summary_record_67b = {
    "generated_at": GENERATED_AT, "git_branch": GIT_BRANCH, "git_commit": GIT_COMMIT,
    "rsna_available": RSNA_AVAILABLE,
    "absolute_level_reference_available": absolute_level_reference_available,
    "dataset_inventory_summary": dataset_inventory_summary,
    "level_schema_audit": level_schema_audit,
    "condition_schema_audit": condition_schema_audit,
    "coordinate_schema_audit": coordinate_schema_audit,
    "sag_t2_canal_stenosis_audit": sag_t2_canal_stenosis_audit,
    "gates": gate_status_67b,
    "decision": decision_67b, "blocker_reason": blocker_reason, "blocking_gates": blocking_gates_67b,
    "absolute_level_training_dataset_ready": ABSOLUTE_LEVEL_TRAINING_DATASET_READY,
    "training_performed": False, "checkpoint_modified": False,
    "rsna_internal_test_locked": RSNA_INTERNAL_TEST_LOCKED,
    "split_counts": rsna_split_counts,
    "warnings": warnings, "limitations": limitations,
}
safe_write_text(ANCHOR_DIR / "post_e50_67B_summary.json", json.dumps(summary_record_67b, indent=2, default=_json_default, ensure_ascii=False))

if DRIVE_RESULTS_DIR is not None:
    for target_dir in (DRIVE_RESULTS_DIR, DRIVE_METRICS_DIR, DRIVE_FIGURES_DIR):
        target_dir.mkdir(parents=True, exist_ok=True)
    if len(annotation_geometry_rows):
        pd.DataFrame(annotation_geometry_rows).to_csv(DRIVE_METRICS_DIR / "post_e50_67B_annotation_geometry.csv", index=False)
    if len(rsna_series_inventory_rows):
        pd.DataFrame(rsna_series_inventory_rows).to_csv(DRIVE_RESULTS_DIR / "rsna_series_inventory.csv", index=False)
    if len(sag_t2_canal_stenosis_reference_rows):
        pd.DataFrame(sag_t2_canal_stenosis_reference_rows).to_csv(DRIVE_RESULTS_DIR / "sag_t2_canal_stenosis_reference_table.csv", index=False)
    print("Drive runtime outputs written under: results/metrics/figures/post_e50/67B (relative to PFI_DRIVE_ROOT)")
else:
    warnings.append("Drive output directories not configured in this run (not in Colab / PFI_DRIVE_ROOT unset).")

print("Local repo artifacts written under artifacts/post_e50/absolute_level_anchor/:")
for f in sorted(ANCHOR_DIR.rglob("*")):
    if f.is_file():
        print(" -", f.relative_to(REPO_ROOT))


Drive runtime outputs written under: results/metrics/figures/post_e50/67B (relative to PFI_DRIVE_ROOT)
Local repo artifacts written under artifacts/post_e50/absolute_level_anchor/:
 - artifacts/post_e50/absolute_level_anchor/post_e50_67B_environment.json
 - artifacts/post_e50/absolute_level_anchor/post_e50_67B_quality_gates.json
 - artifacts/post_e50/absolute_level_anchor/post_e50_67B_smoke_cohort.json
 - artifacts/post_e50/absolute_level_anchor/post_e50_67B_split_manifest.json
 - artifacts/post_e50/absolute_level_anchor/post_e50_67B_summary.json
 - artifacts/post_e50/absolute_level_anchor/rsna_dataset_inventory.json
 - artifacts/post_e50/absolute_level_anchor/rsna_level_reference_audit.json
 - artifacts/post_e50/absolute_level_anchor/rsna_series_inventory.csv
 - artifacts/post_e50/absolute_level_anchor/sag_t2_canal_stenosis_reference_table.csv


## Reporte (Markdown)


In [24]:
report_lines_67b = []
report_lines_67b.append("# Post-E50 Absolute Lumbar Level Anchor (RSNA) -- Stage A + Stage B")
report_lines_67b.append("")
report_lines_67b.append("## Roadmap")
report_lines_67b.append("")
report_lines_67b.append(
    "67A = SPIDER relative localization validation (closed). 67B (this notebook) = absolute lumbar "
    "level anchor. 67C = axial cluster-level pairing (future, blocked until 67B produces an anchor)."
)
report_lines_67b.append("")
report_lines_67b.append("## Objective")
report_lines_67b.append("")
report_lines_67b.append(
    "Resolve, reproducibly and evaluably: predicted relative disc instances -> absolute lumbar "
    "levels (L1-L2...L5-S1), without using ground truth to decide the answer during inference. "
    "GT is used exclusively for training/evaluation."
)
report_lines_67b.append("")
report_lines_67b.append("## Execution status (this run)")
report_lines_67b.append("")
report_lines_67b.append(f"- RSNA_AVAILABLE: `{RSNA_AVAILABLE}`")
report_lines_67b.append(f"- officialTestPresent: `{OFFICIAL_TEST_PRESENT}`, officialTestAccessed: `{OFFICIAL_TEST_ACCESSED}` (never accessed, structurally)")
if not RSNA_AVAILABLE:
    report_lines_67b.append("- Stage A / local validation: RSNA data was not accessed in this run (expected outside Colab / without Drive mounted). This is a Colab-ready design, stopped before real execution.")
report_lines_67b.append("")
report_lines_67b.append("## Stage A -- dataset structure and schema audit")
report_lines_67b.append("")
report_lines_67b.append(f"- GATE A (dataset structure): `{gate_status_67b['GATE_A_RSNA_dataset_structure']}`")
report_lines_67b.append(f"- GATE B (absolute level reference availability): `{absolute_level_reference_available}`")
report_lines_67b.append(f"- GATE C (split leakage, study_id-level): `{gate_status_67b['GATE_C_split_leakage']}`")
report_lines_67b.append("")
report_lines_67b.append(json.dumps({"csv_schema_report": csv_schema_report, "dataset_inventory_summary": dataset_inventory_summary}, indent=2, default=str) if csv_schema_report else "_RSNA not available in this run -- no real schema to report._")
report_lines_67b.append("")
report_lines_67b.append("## Stage B -- coordinate geometry design")
report_lines_67b.append("")
report_lines_67b.append(f"- coordinate_mapping_formula_selftest (method-level, synthetic pydicom self-test): `{coordinate_mapping_formula_selftest}`")
report_lines_67b.append(f"- simpleitk_physical_crosscheck (method-level, independent SimpleITK geometry engine): `{simpleitk_physical_crosscheck}`")
report_lines_67b.append(
    "- **These two sub_audits confirm the METHOD is correct, not parity against real RSNA DICOM.** "
    "GATE_D and GATE_E never inherit a PASS from them -- they require real data."
)
report_lines_67b.append(f"- GATE D (series/annotation-to-DICOM mapping, real data): `{gate_status_67b['GATE_D_series_annotation_mapping']}`")
report_lines_67b.append(f"- GATE E (coordinate physical parity, real data): `{gate_status_67b['GATE_E_coordinate_physical_parity']}`")
report_lines_67b.append(
    "- `dicom_pixel_to_patient_xyz` reuses the same class of DICOM pixel-to-patient-physical formula "
    "as Notebook 66/67 -- no new geometric convention was invented."
)
report_lines_67b.append("")
report_lines_67b.append("## Stage C / D -- NOT executed in this run (explicit scope boundary)")
report_lines_67b.append("")
report_lines_67b.append(
    "Frozen 67A baseline execution on RSNA (Stage C) and the absolute-anchor training-dataset design "
    "(Stage D) are documented and gate-scaffolded but intentionally not executed in this iteration. "
    f"Expected checkpoint SHA-256 (reference only): `{EXPECTED_CHECKPOINT_SHA256}`."
)
report_lines_67b.append("")
report_lines_67b.append("## Decision")
report_lines_67b.append("")
report_lines_67b.append(f"- Decision: `{decision_67b}`")
report_lines_67b.append(f"- Blocker reason: `{blocker_reason}`")
report_lines_67b.append(f"- Blocking gates: `{blocking_gates_67b}`")
report_lines_67b.append(f"- ABSOLUTE_LEVEL_TRAINING_DATASET_READY: `{ABSOLUTE_LEVEL_TRAINING_DATASET_READY}`")
report_lines_67b.append("")
report_lines_67b.append("## Quality gates")
report_lines_67b.append("")
report_lines_67b.append(gates_67b_df.to_markdown(index=False))
report_lines_67b.append("")
report_lines_67b.append("## What this notebook does NOT do")
report_lines_67b.append("")
for item in [
    "no training performed (segmenter, classifier, or any model)",
    "no checkpoint modified",
    "no Notebook 67 / 67A modification",
    "no axial cluster-level pairing",
    "no cross-frame registration",
    "no pathology grading",
    "no official RSNA test access (officialTestAccessed=False, structurally enforced)",
    "no SPIDER test access",
    "no assumption that a study must have exactly 5 visible levels",
]:
    report_lines_67b.append(f"- {item}")
report_lines_67b.append("")

report_text_67b = "\n".join(report_lines_67b)
safe_write_text(REPORT_DIR / "post_e50_absolute_level_anchor_rsna_report.md", report_text_67b)
print(f"Report written: {(REPORT_DIR / 'post_e50_absolute_level_anchor_rsna_report.md').relative_to(REPO_ROOT)}")


Report written: reports/post_e50/post_e50_absolute_level_anchor_rsna_report.md


## EXPERIMENT STATUS


In [25]:
EXECUTION_SECONDS_67B = time.time() - EXECUTION_START
gpu_label_67b = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"

status_block_67b = f'''EXPERIMENT STATUS

Experiment:
Post-E50 Absolute Lumbar Level Anchor (RSNA) -- Stage A + Stage B

Training performed:
NO

Environment:
{"COLAB" if IN_COLAB else "LOCAL"}

GPU:
{gpu_label_67b}

RSNA root:
{"CONFIGURED" if RSNA_AVAILABLE else "MISSING"}

Official test present:
{OFFICIAL_TEST_PRESENT}

Official test accessed:
{OFFICIAL_TEST_ACCESSED}

GATE_A_RSNA_dataset_structure:
{gate_status_67b["GATE_A_RSNA_dataset_structure"]}

GATE_B_absolute_level_reference:
{gate_status_67b["GATE_B_absolute_level_reference"]}

GATE_C_split_leakage:
{gate_status_67b["GATE_C_split_leakage"]}

GATE_D_series_annotation_mapping:
{gate_status_67b["GATE_D_series_annotation_mapping"]}

GATE_E_coordinate_physical_parity:
{gate_status_67b["GATE_E_coordinate_physical_parity"]}

GATE_F_frozen_67A_execution_on_RSNA:
{gate_status_67b["GATE_F_frozen_67A_execution_on_RSNA"]}

GATE_G_pred_to_absolute_GT_matching:
{gate_status_67b["GATE_G_pred_to_absolute_GT_matching"]}

GATE_H_training_dataset_readiness:
{gate_status_67b["GATE_H_training_dataset_readiness"]}

GATE_I_privacy:
{gate_status_67b["GATE_I_privacy"]}

Decision:
{decision_67b}

Blocker reason:
{blocker_reason}

RSNA_INTERNAL_TEST_LOCKED:
{RSNA_INTERNAL_TEST_LOCKED}

Execution seconds:
{EXECUTION_SECONDS_67B:.1f}

Warnings:
{chr(10).join(warnings) if warnings else "(none)"}
'''

print(status_block_67b)


EXPERIMENT STATUS

Experiment:
Post-E50 Absolute Lumbar Level Anchor (RSNA) -- Stage A + Stage B

Training performed:
NO

Environment:
COLAB

GPU:
none (CPU)

RSNA root:
CONFIGURED

Official test present:
False

Official test accessed:
False

GATE_A_RSNA_dataset_structure:
PASS

GATE_B_absolute_level_reference:
YES

GATE_C_split_leakage:
PASS

GATE_D_series_annotation_mapping:
PASS

GATE_E_coordinate_physical_parity:
PASS

GATE_F_frozen_67A_execution_on_RSNA:
NOT_RUN

GATE_G_pred_to_absolute_GT_matching:
NOT_RUN

GATE_H_training_dataset_readiness:
PARTIAL

GATE_I_privacy:
PASS

Decision:
PARTIAL

Blocker reason:
SEE_BLOCKING_GATES

RSNA_INTERNAL_TEST_LOCKED:
True

Execution seconds:
5358.7

Warnings:
STAGE C (frozen 67A baseline EXECUTION on RSNA) intentionally NOT run in this notebook -- preflight/cohort-selection only, per explicit instruction. Review Stage A/B results first.

